In [ ]:
# ===============================
# Step 1: Environment Setup
# ===============================

# Core
import os
import random
import numpy as np
import pandas as pd

# Visualization (later use, safe to import now)
import matplotlib.pyplot as plt
import seaborn as sns

# ML / NLP
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:
# ===============================
# Step 1: Load Datasets
# ===============================

BASE_URL = "https://huggingface.co/datasets/ourafla/Mental-Health_Text-Classification_Dataset/resolve/main/"

train_path = BASE_URL + "mental_heath_unbanlanced.csv"
test_path  = BASE_URL + "mental_health_combined_test.csv"
feature_path = BASE_URL + "mental_health_feature_engineered.csv"

# Load CSVs
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
feature_df = pd.read_csv(feature_path)

print("Datasets loaded successfully.")


Datasets loaded successfully.


In [11]:
# Zip the task1_models folder for download
import shutil
import os

print("Creating zip file of task1_models folder...")

# Create zip file
shutil.make_archive('task1_models', 'zip', '.', 'task1_models')

print("✓ Zip file created: task1_models.zip")
print(f"✓ File size: {os.path.getsize('task1_models.zip') / (1024*1024):.2f} MB")
print("\nTo download: Right-click on 'task1_models.zip' in the Files panel and select Download")

Creating zip file of task1_models folder...
✓ Zip file created: task1_models.zip
✓ File size: 387.61 MB

To download: Right-click on 'task1_models.zip' in the Files panel and select Download


In [ ]:
# ================================================================
# HOW TO USE SAVED MODELS IN FUTURE SESSIONS (WITHOUT RETRAINING)
# ================================================================

"""
STEP 1: UPLOAD THE ZIP FILE
---------------------------
1. In a new Colab session, go to Files panel (left sidebar)
2. Click the upload button
3. Upload 'task1_models.zip' (387 MB)
4. Wait for upload to complete

STEP 2: UNZIP THE FOLDER
-------------------------
"""
import shutil
import os

# Unzip the uploaded file
shutil.unpack_archive('task1_models.zip', '.', 'zip')
print("✓ Models extracted to ./task1_models/")

"""
STEP 3: RELOAD THE MODELS
--------------------------
"""
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pickle

# A) Reload BERT model (for predictions)
print("\nLoading BERT model...")
bert_model = AutoModelForSequenceClassification.from_pretrained(
    './task1_models/bert_mental_health_classifier'
)
bert_tokenizer = AutoTokenizer.from_pretrained(
    './task1_models/bert_mental_health_classifier'
)
print("✓ BERT model loaded")

# B) Reload Baseline models (optional)
print("\nLoading baseline models...")
with open('./task1_models/baseline_tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
with open('./task1_models/baseline_lr_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)
print("✓ Baseline models loaded")

# C) Reload label mappings
with open('./task1_models/label_mappings.pkl', 'rb') as f:
    label_data = pickle.load(f)
    LABELS = label_data['LABELS']
    label2id = label_data['label2id']
    id2label = label_data['id2label']
print("✓ Label mappings loaded")
print(f"   Labels: {LABELS}")

"""
STEP 4: MAKE PREDICTIONS ON NEW TEXT
-------------------------------------
"""
import torch
import numpy as np
from scipy.special import softmax

def predict_mental_state(text, model=bert_model, tokenizer=bert_tokenizer):
    """
    Predict mental health state for a given text.

    Returns:
        - predicted_label: str (Normal/Anxiety/Depression/Suicidal)
        - probabilities: dict with confidence for each class
    """
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)

    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.numpy()[0]

    # Get probabilities
    probs = softmax(logits)
    predicted_id = np.argmax(probs)
    predicted_label = id2label[predicted_id]

    # Format probabilities
    prob_dict = {label: float(probs[label2id[label]]) for label in LABELS}

    return predicted_label, prob_dict

# Example usage
print("\n" + "="*60)
print("EXAMPLE PREDICTIONS")
print("="*60)

test_texts = [
    "I'm feeling great today! Everything is going well.",
    "I'm worried about my exams and can't stop thinking about it.",
    "I feel hopeless and nothing seems to matter anymore."
]

for i, text in enumerate(test_texts, 1):
    label, probs = predict_mental_state(text)
    print(f"\nText {i}: \"{text[:50]}...\"")
    print(f"Prediction: {label}")
    print(f"Confidence: {probs[label]:.2%}")
    print(f"All probabilities: {', '.join([f'{k}: {v:.2%}' for k, v in probs.items()])}")

print("\n" + "="*60)
print("MODELS READY FOR USE!")
print("="*60)
print("\n✓ You can now use predict_mental_state(text) for any new text")
print("✓ No need to retrain - saves 89 minutes of GPU time!")
print("✓ Use this for Task 2 and Task 3 in your capstone project")

In [12]:
# ===============================
# Export All Results & Outputs
# ===============================

import pandas as pd
import json
from datetime import datetime
import os

print("="*70)
print("EXPORTING ALL TASK 1 RESULTS")
print("="*70)

# Create results directory
RESULTS_DIR = "./task1_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"\n[1/6] Creating results directory: {RESULTS_DIR}")

# 1. Export Baseline Results
print("\n[2/6] Exporting baseline evaluation results...")
baseline_results = {
    'model': 'Baseline (TF-IDF + Logistic Regression)',
    'validation': {
        'accuracy': 0.7872,
        'f1_macro': 0.7715,
        'f1_normal': 0.9172,
        'f1_anxiety': 0.7919,
        'f1_depression': 0.6820,
        'f1_suicidal': 0.6987
    },
    'test': {
        'accuracy': 0.7147,
        'f1_macro': 0.7126,
        'f1_normal': 0.8040,
        'f1_anxiety': 0.7625,
        'f1_depression': 0.5690,
        'f1_suicidal': 0.7149
    }
}

with open(f'{RESULTS_DIR}/baseline_metrics.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)
print("  ✓ baseline_metrics.json")

# 2. Export BERT Results
print("\n[3/6] Exporting BERT evaluation results...")
bert_results = {
    'model': 'BERT (bert-base-uncased)',
    'training': {
        'epochs': 3,
        'training_time_minutes': 89.09,
        'learning_rate': 2e-5,
        'batch_size': 16
    },
    'validation': {
        'accuracy': 0.8587,
        'f1_macro': 0.8528,
        'f1_normal': 0.9603,
        'f1_anxiety': 0.8940,
        'f1_depression': 0.7837,
        'f1_suicidal': 0.7731
    },
    'test': {
        'accuracy': bert_test_acc,
        'f1_macro': bert_test_f1_macro,
        'f1_normal': float(bert_test_f1_per_class[0]),
        'f1_anxiety': float(bert_test_f1_per_class[1]),
        'f1_depression': float(bert_test_f1_per_class[2]),
        'f1_suicidal': float(bert_test_f1_per_class[3])
    }
}

with open(f'{RESULTS_DIR}/bert_metrics.json', 'w') as f:
    json.dump(bert_results, f, indent=2)
print("  ✓ bert_metrics.json")

# 3. Export Comparison Table
print("\n[4/6] Exporting baseline vs BERT comparison...")
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Macro', 'F1 Normal', 'F1 Anxiety', 'F1 Depression', 'F1 Suicidal'],
    'Baseline_Val': [0.7878, 0.7724, 0.9172, 0.7919, 0.6820, 0.6987],
    'BERT_Val': [0.8587, 0.8528, 0.9603, 0.8940, 0.7837, 0.7731],
    'Baseline_Test': [0.7147, 0.7126, 0.8040, 0.7625, 0.5690, 0.7149],
    'BERT_Test': [
        bert_test_acc,
        bert_test_f1_macro,
        float(bert_test_f1_per_class[0]),
        float(bert_test_f1_per_class[1]),
        float(bert_test_f1_per_class[2]),
        float(bert_test_f1_per_class[3])
    ]
})

comparison_df['Improvement_Val'] = comparison_df['BERT_Val'] - comparison_df['Baseline_Val']
comparison_df['Improvement_Test'] = comparison_df['BERT_Test'] - comparison_df['Baseline_Test']
comparison_df['Improvement_Val_%'] = (comparison_df['Improvement_Val'] * 100).round(2)
comparison_df['Improvement_Test_%'] = (comparison_df['Improvement_Test'] * 100).round(2)

comparison_df.to_csv(f'{RESULTS_DIR}/model_comparison.csv', index=False)
print("  ✓ model_comparison.csv")

# 4. Export Confusion Matrices
print("\n[5/6] Exporting confusion matrices...")
confusion_data = {
    'baseline_val': confusion_matrix(y_val, y_val_pred).tolist(),
    'baseline_test': confusion_matrix(y_test, y_test_pred).tolist(),
    'bert_test': confusion_matrix(test_labels, test_preds).tolist(),
    'labels': LABELS
}

with open(f'{RESULTS_DIR}/confusion_matrices.json', 'w') as f:
    json.dump(confusion_data, f, indent=2)
print("  ✓ confusion_matrices.json")

# 5. Export Summary Report
print("\n[6/6] Creating summary report...")
summary_report = f"""TASK 1: MENTAL-STATE CLASSIFICATION - RESULTS SUMMARY
{'='*70}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

DATASET INFORMATION:
- Training samples: 44,650
- Validation samples: 4,962
- Test samples: 992
- Classes: Normal, Anxiety, Depression, Suicidal

MODEL 1: BASELINE (TF-IDF + LOGISTIC REGRESSION)
{'='*70}
Validation Performance:
  - Accuracy: {baseline_results['validation']['accuracy']:.4f}
  - F1 Macro: {baseline_results['validation']['f1_macro']:.4f}
  - F1 Suicidal: {baseline_results['validation']['f1_suicidal']:.4f}

Test Performance:
  - Accuracy: {baseline_results['test']['accuracy']:.4f}
  - F1 Macro: {baseline_results['test']['f1_macro']:.4f}
  - F1 Suicidal: {baseline_results['test']['f1_suicidal']:.4f}

MODEL 2: BERT (bert-base-uncased)
{'='*70}
Training:
  - Epochs: 3
  - Training Time: 89.09 minutes
  - GPU: T4

Validation Performance:
  - Accuracy: {bert_results['validation']['accuracy']:.4f}
  - F1 Macro: {bert_results['validation']['f1_macro']:.4f}
  - F1 Suicidal: {bert_results['validation']['f1_suicidal']:.4f}

Test Performance:
  - Accuracy: {bert_test_acc:.4f}
  - F1 Macro: {bert_test_f1_macro:.4f}
  - F1 Suicidal: {float(bert_test_f1_per_class[3]):.4f}

KEY IMPROVEMENTS (BERT vs Baseline on Test Set):
{'='*70}
  - Overall F1: +{(bert_test_f1_macro - 0.7126)*100:.1f} percentage points
  - Suicidal F1: +{(float(bert_test_f1_per_class[3]) - 0.7149)*100:.1f} percentage points
  - Depression F1: +{(float(bert_test_f1_per_class[2]) - 0.5690)*100:.1f} percentage points

CRITICAL FINDINGS:
{'='*70}
1. BERT achieves 88.6% test F1 (macro) vs 71.3% baseline
2. Suicidal class detection improved from 71.5% to 93.0%
3. Depression class detection improved from 56.9% to 80.2%
4. Lower false negatives = fewer missed crisis cases
5. Suitable for operational triage priority scoring (Task 3)

FILES SAVED:
{'='*70}
1. baseline_metrics.json - Baseline model performance
2. bert_metrics.json - BERT model performance
3. model_comparison.csv - Side-by-side comparison table
4. confusion_matrices.json - All confusion matrices
5. test_predictions_with_probs.csv - Test set predictions
6. summary_report.txt - This report

NEXT STEPS:
{'='*70}
- Task 2: C-SSRS Suicide Risk Severity Scoring
- Task 3: Operational Triage Priority Scoring
- Use saved BERT model for predictions without retraining
"""

with open(f'{RESULTS_DIR}/summary_report.txt', 'w') as f:
    f.write(summary_report)
print("  ✓ summary_report.txt")

# Also copy test predictions to results folder
import shutil
shutil.copy('./task1_models/test_predictions_with_probs.csv',
            f'{RESULTS_DIR}/test_predictions_with_probs.csv')
print("  ✓ test_predictions_with_probs.csv (copied)")

print("\n" + "="*70)
print("ALL RESULTS EXPORTED SUCCESSFULLY!")
print("="*70)
print(f"\nLocation: {RESULTS_DIR}/")
print("\nFiles created:")
for file in os.listdir(RESULTS_DIR):
    size = os.path.getsize(f'{RESULTS_DIR}/{file}')
    print(f"  - {file} ({size:,} bytes)")

print("\n💡 TIP: Download the entire 'task1_results' folder for your capstone report!")
print("   Right-click on 'task1_results' in Files panel → Download")

EXPORTING ALL TASK 1 RESULTS

[1/6] Creating results directory: ./task1_results

[2/6] Exporting baseline evaluation results...
  ✓ baseline_metrics.json

[3/6] Exporting BERT evaluation results...
  ✓ bert_metrics.json

[4/6] Exporting baseline vs BERT comparison...
  ✓ model_comparison.csv

[5/6] Exporting confusion matrices...
  ✓ confusion_matrices.json

[6/6] Creating summary report...
  ✓ summary_report.txt
  ✓ test_predictions_with_probs.csv (copied)

ALL RESULTS EXPORTED SUCCESSFULLY!

Location: ./task1_results/

Files created:
  - baseline_metrics.json (401 bytes)
  - summary_report.txt (2,196 bytes)
  - test_predictions_with_probs.csv (850,329 bytes)
  - confusion_matrices.json (782 bytes)
  - bert_metrics.json (583 bytes)
  - model_comparison.csv (729 bytes)

💡 TIP: Download the entire 'task1_results' folder for your capstone report!
   Right-click on 'task1_results' in Files panel → Download


In [13]:
# Zip the task1_results folder for easy download
import shutil

print("Creating zip file of task1_results folder...")

# Create zip file
shutil.make_archive('task1_results', 'zip', '.', 'task1_results')

print("✓ Zip file created: task1_results.zip")
print(f"✓ File size: {os.path.getsize('task1_results.zip') / 1024:.2f} KB")
print("\n💡 To download: Right-click on 'task1_results.zip' in the Files panel and select Download")

Creating zip file of task1_results folder...
✓ Zip file created: task1_results.zip
✓ File size: 314.81 KB

💡 To download: Right-click on 'task1_results.zip' in the Files panel and select Download


In [ ]:
# ===============================
# Dataset Inspection
# ===============================

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns)

print("\nLabel distribution (Train):")
print(train_df["status"].value_counts())

print("\nLabel distribution (Test):")
print(test_df["status"].value_counts())


Train shape: (49612, 3)
Test shape: (992, 2)

Train columns:
Index(['Unique_ID', 'text', 'status'], dtype='object')

Label distribution (Train):
status
Normal        18391
Depression    14506
Suicidal      11212
Anxiety        5503
Name: count, dtype: int64

Label distribution (Test):
status
Anxiety       248
Depression    248
Normal        248
Suicidal      248
Name: count, dtype: int64


In [ ]:
# ===============================
# Step 2: Standardize + Validate + Split (Train -> Train/Val)
# ===============================

# 1) Keep only required columns and ensure consistent naming
train_df = train_df[["text", "status"]].copy()
test_df  = test_df[["text", "status"]].copy()

# 2) Basic quality checks
def basic_checks(df, name):
    print(f"\n--- {name} checks ---")
    print("Shape:", df.shape)
    print("Null text:", df["text"].isna().sum())
    print("Null status:", df["status"].isna().sum())
    print("Unique labels:", df["status"].unique())
    print("Label counts:\n", df["status"].value_counts())

basic_checks(train_df, "TRAIN")
basic_checks(test_df, "TEST")



--- TRAIN checks ---
Shape: (49612, 2)
Null text: 0
Null status: 0
Unique labels: ['Anxiety' 'Normal' 'Depression' 'Suicidal']
Label counts:
 status
Normal        18391
Depression    14506
Suicidal      11212
Anxiety        5503
Name: count, dtype: int64

--- TEST checks ---
Shape: (992, 2)
Null text: 0
Null status: 0
Unique labels: ['Anxiety' 'Depression' 'Normal' 'Suicidal']
Label counts:
 status
Anxiety       248
Depression    248
Normal        248
Suicidal      248
Name: count, dtype: int64


In [ ]:
# ===============================
# Step 2: Standardize + Encode + Train/Val Split
# ===============================

from sklearn.model_selection import train_test_split

# Keep only required columns (defensive)
train_df = train_df[["text", "status"]].copy()
test_df  = test_df[["text", "status"]].copy()

# Basic type cleanup
train_df["text"] = train_df["text"].astype(str)
test_df["text"]  = test_df["text"].astype(str)

train_df["status"] = train_df["status"].astype(str).str.strip()
test_df["status"]  = test_df["status"].astype(str).str.strip()

# Canonical label order (fixes consistency across all downstream code)
LABELS = ["Normal", "Anxiety", "Depression", "Suicidal"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

# Validate label set (hard fail if unexpected labels appear)
train_labels_set = set(train_df["status"].unique())
test_labels_set  = set(test_df["status"].unique())
expected_set = set(LABELS)

assert train_labels_set.issubset(expected_set), f"Unexpected labels in train: {train_labels_set - expected_set}"
assert test_labels_set.issubset(expected_set),  f"Unexpected labels in test: {test_labels_set - expected_set}"

# Encode labels
train_df["label"] = train_df["status"].map(label2id)
test_df["label"]  = test_df["status"].map(label2id)

# Create validation split from TRAIN only (stratified)
train_split_df, val_df = train_test_split(
    train_df,
    test_size=0.10,          # 10% validation (standard)
    random_state=SEED,
    stratify=train_df["label"]
)

# Reset indices for cleanliness
train_split_df = train_split_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("--- SPLIT SUMMARY ---")
print("Train split:", train_split_df.shape)
print("Val split:  ", val_df.shape)
print("Test:       ", test_df.shape)

print("\nTrain label distribution:")
print(train_split_df["status"].value_counts())

print("\nVal label distribution:")
print(val_df["status"].value_counts())

print("\nTest label distribution:")
print(test_df["status"].value_counts())


--- SPLIT SUMMARY ---
Train split: (44650, 3)
Val split:   (4962, 3)
Test:        (992, 3)

Train label distribution:
status
Normal        16551
Depression    13055
Suicidal      10091
Anxiety        4953
Name: count, dtype: int64

Val label distribution:
status
Normal        1840
Depression    1451
Suicidal      1121
Anxiety        550
Name: count, dtype: int64

Test label distribution:
status
Anxiety       248
Depression    248
Normal        248
Suicidal      248
Name: count, dtype: int64


In [ ]:
# ===============================
# Step 3: HF Datasets + Tokenization
# ===============================

# Reinstall pyarrow first to resolve potential binary incompatibility
!pip uninstall -y pyarrow
!pip install -q pyarrow

!pip -q install -U transformers datasets accelerate evaluate

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_CHECKPOINT = "bert-base-uncased"  # later you can switch to a mental-health domain model
MAX_LENGTH = 256                       # safe default for social posts; adjust later if needed

# 1) Convert pandas -> HF Dataset
hf_train = Dataset.from_pandas(train_split_df[["text", "label"]], preserve_index=False)
hf_val   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
hf_test  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

dataset = DatasetDict({
    "train": hf_train,
    "validation": hf_val,
    "test": hf_test
})

print(dataset)

# 2) Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

# 3) Tokenization function
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

# 4) Tokenize all splits
tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])

# 5) Dynamic padding collator (pads each batch to max length within batch)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\nTokenized features:", tokenized["train"].features)
print("Example tokenized row keys:", tokenized["train"][0].keys())


Found existing installation: pyarrow 22.0.0
Uninstalling pyarrow-22.0.0:
  Successfully uninstalled pyarrow-22.0.0
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 44650
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 4962
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 992
    })
})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/44650 [00:00<?, ? examples/s]

Map:   0%|          | 0/4962 [00:00<?, ? examples/s]

Map:   0%|          | 0/992 [00:00<?, ? examples/s]


Tokenized features: {'label': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Example tokenized row keys: dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:
# ===============================
# Step 4: Baseline Model (TF-IDF + Logistic Regression)
# ===============================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import time

print("=" * 60)
print("BASELINE MODEL: TF-IDF + Logistic Regression")
print("=" * 60)

# 1) TF-IDF Vectorization
print("\n[1/4] Vectorizing text with TF-IDF...")
start_time = time.time()

vectorizer = TfidfVectorizer(
    max_features=5000,     # Keep top 5000 features
    ngram_range=(1, 2),    # Unigrams + bigrams
    min_df=2,              # Ignore terms that appear in < 2 documents
    max_df=0.95,           # Ignore terms that appear in > 95% of documents
    strip_accents='unicode',
    lowercase=True
)

X_train_tfidf = vectorizer.fit_transform(train_split_df["text"])
X_val_tfidf = vectorizer.transform(val_df["text"])
X_test_tfidf = vectorizer.transform(test_df["text"])

y_train = train_split_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

print(f"   Train shape: {X_train_tfidf.shape}")
print(f"   Val shape:   {X_val_tfidf.shape}")
print(f"   Test shape:  {X_test_tfidf.shape}")
print(f"   Time: {time.time() - start_time:.2f}s")

# 2) Train Logistic Regression
print("\n[2/4] Training Logistic Regression (class_weight='balanced')...")
start_time = time.time()

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handle class imbalance
    random_state=SEED,
    solver='lbfgs',
    n_jobs=-1  # Use all CPU cores
)

lr_model.fit(X_train_tfidf, y_train)
print(f"   Training completed in {time.time() - start_time:.2f}s")

# 3) Evaluate on Validation Set
print("\n[3/4] Evaluating on VALIDATION set...")
y_val_pred = lr_model.predict(X_val_tfidf)
val_acc = accuracy_score(y_val, y_val_pred)
val_f1_macro = f1_score(y_val, y_val_pred, average='macro')

print(f"   Validation Accuracy: {val_acc:.4f}")
print(f"   Validation Macro F1: {val_f1_macro:.4f}")
print("\n   Classification Report (Validation):")
print(classification_report(y_val, y_val_pred, target_names=LABELS, digits=4))

print("\n   Confusion Matrix (Validation):")
print(confusion_matrix(y_val, y_val_pred))

# 4) Evaluate on Test Set
print("\n[4/4] Evaluating on TEST set...")
y_test_pred = lr_model.predict(X_test_tfidf)
test_acc = accuracy_score(y_test, y_test_pred)
test_f1_macro = f1_score(y_test, y_test_pred, average='macro')

print(f"   Test Accuracy: {test_acc:.4f}")
print(f"   Test Macro F1: {test_f1_macro:.4f}")
print("\n   Classification Report (Test):")
print(classification_report(y_test, y_test_pred, target_names=LABELS, digits=4))

print("\n   Confusion Matrix (Test):")
print(confusion_matrix(y_test, y_test_pred))

print("\n" + "=" * 60)
print("BASELINE MODEL COMPLETE")
print(f"Validation F1 (macro): {val_f1_macro:.4f}")
print(f"Test F1 (macro):       {test_f1_macro:.4f}")
print("=" * 60)

BASELINE MODEL: TF-IDF + Logistic Regression

[1/4] Vectorizing text with TF-IDF...
   Train shape: (44650, 5000)
   Val shape:   (4962, 5000)
   Test shape:  (992, 5000)
   Time: 22.82s

[2/4] Training Logistic Regression (class_weight='balanced')...
   Training completed in 12.65s

[3/4] Evaluating on VALIDATION set...
   Validation Accuracy: 0.7872
   Validation Macro F1: 0.7715

   Classification Report (Validation):
              precision    recall  f1-score   support

      Normal     0.9056    0.9277    0.9165      1840
     Anxiety     0.7108    0.8891    0.7900       550
  Depression     0.7681    0.6141    0.6825      1451
    Suicidal     0.6664    0.7306    0.6970      1121

    accuracy                         0.7872      4962
   macro avg     0.7627    0.7904    0.7715      4962
weighted avg     0.7897    0.7872    0.7845      4962


   Confusion Matrix (Validation):
[[1707   52   40   41]
 [  21  489   30   10]
 [  80  121  891  359]
 [  77   26  199  819]]

[4/4] Evalu

In [ ]:
# ===============================
# Step 5: BERT Fine-Tuning
# ===============================

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from scipy.special import softmax

print("=" * 60)
print("BERT FINE-TUNING")
print("=" * 60)

# 1) Load BERT model
print("\n[1/5] Loading BERT model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

print(f"   Model: {MODEL_CHECKPOINT}")
print(f"   Num labels: 4")
print(f"   Model parameters: {model.num_parameters():,}")

# 2) Define training arguments
print("\n[2/5] Configuring training parameters...")
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    seed=SEED,
    report_to="none"  # Disable wandb
)

print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Best model metric: {training_args.metric_for_best_model}")

# 3) Define evaluation metrics
print("\n[3/5] Setting up evaluation metrics...")
metric_f1 = evaluate.load("f1")
metric_accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1_macro = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"]

    # Per-class F1
    f1_per_class = metric_f1.compute(predictions=predictions, references=labels, average=None)["f1"]

    return {
        "f1_macro": f1_macro,
        "accuracy": accuracy,
        "f1_normal": f1_per_class[0],
        "f1_anxiety": f1_per_class[1],
        "f1_depression": f1_per_class[2],
        "f1_suicidal": f1_per_class[3]
    }

print("   Metrics: F1 (macro), Accuracy, Per-class F1")

# 4) Create Trainer
print("\n[4/5] Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

print("   Trainer initialized successfully")

# 5) Train the model
print("\n[5/5] Starting training...")
print("=" * 60)
start_time = time.time()

train_result = trainer.train()

training_time = time.time() - start_time
print("=" * 60)
print(f"\nTraining completed in {training_time:.2f}s ({training_time/60:.2f} minutes)")
print(f"Best validation F1 (macro): {train_result.metrics.get('eval_f1_macro', 'N/A')}")

print("\n" + "=" * 60)
print("BERT TRAINING COMPLETE")
print("=" * 60)

BERT FINE-TUNING

[1/5] Loading BERT model...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Model: bert-base-uncased
   Num labels: 4
   Model parameters: 109,485,316

[2/5] Configuring training parameters...
   Learning rate: 2e-05
   Batch size: 16
   Epochs: 3
   Best model metric: f1_macro

[3/5] Setting up evaluation metrics...


   Metrics: F1 (macro), Accuracy, Per-class F1

[4/5] Initializing Trainer...


/tmp/ipython-input-1290455500.py:78: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


   Trainer initialized successfully

[5/5] Starting training...


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Normal,F1 Anxiety,F1 Depression,F1 Suicidal
1,0.385900,0.380224,0.834922,0.840992,0.953762,0.884448,0.739012,0.762466
2,0.313300,0.381302,0.846830,0.852277,0.955689,0.890080,0.761103,0.780449
3,0.235600,0.417950,0.852799,0.858726,0.960307,0.894034,0.783717,0.773138



Training completed in 5345.39s (89.09 minutes)
Best validation F1 (macro): N/A

BERT TRAINING COMPLETE


In [ ]:
# ===============================
# Step 6: BERT Test Evaluation & Baseline Comparison
# ===============================

import pandas as pd
from scipy.special import softmax

print("=" * 60)
print("BERT TEST SET EVALUATION")
print("=" * 60)

# 1) Get BERT predictions on test set
print("\n[1/4] Running BERT inference on test set...")
test_predictions = trainer.predict(tokenized["test"])
test_logits = test_predictions.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_probs = softmax(test_logits, axis=-1)
test_labels = test_predictions.label_ids

print(f"   Test samples: {len(test_labels)}")
print(f"   Predictions shape: {test_preds.shape}")

# 2) Calculate BERT test metrics
print("\n[2/4] Calculating BERT test metrics...")
bert_test_acc = accuracy_score(test_labels, test_preds)
bert_test_f1_macro = f1_score(test_labels, test_preds, average='macro')
bert_test_f1_per_class = f1_score(test_labels, test_preds, average=None)

print(f"   Test Accuracy: {bert_test_acc:.4f}")
print(f"   Test F1 (macro): {bert_test_f1_macro:.4f}")
print("\n   Classification Report (Test):")
print(classification_report(test_labels, test_preds, target_names=LABELS, digits=4))
print("\n   Confusion Matrix (Test):")
print(confusion_matrix(test_labels, test_preds))

# 3) Create comprehensive comparison table
print("\n[3/4] Creating Baseline vs BERT comparison...")
print("\n" + "=" * 80)
print("FINAL COMPARISON: BASELINE vs BERT")
print("=" * 80)

# Validation metrics comparison
print("\n--- VALIDATION SET PERFORMANCE ---\n")
comparison_val = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Macro', 'F1 Normal', 'F1 Anxiety', 'F1 Depression', 'F1 Suicidal'],
    'Baseline (TF-IDF+LR)': [0.7878, 0.7724, 0.9172, 0.7919, 0.6820, 0.6987],
    'BERT': [0.8587, 0.8528, 0.9603, 0.8940, 0.7837, 0.7731]
})
comparison_val['Improvement'] = comparison_val['BERT'] - comparison_val['Baseline (TF-IDF+LR)']
comparison_val['Improvement (%)'] = (comparison_val['Improvement'] * 100).round(2)
print(comparison_val.to_string(index=False))

# Test metrics comparison
print("\n--- TEST SET PERFORMANCE ---\n")
comparison_test = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Macro', 'F1 Normal', 'F1 Anxiety', 'F1 Depression', 'F1 Suicidal'],
    'Baseline (TF-IDF+LR)': [0.7147, 0.7126, 0.8040, 0.7625, 0.5690, 0.7149],
    'BERT': [
        bert_test_acc,
        bert_test_f1_macro,
        bert_test_f1_per_class[0],
        bert_test_f1_per_class[1],
        bert_test_f1_per_class[2],
        bert_test_f1_per_class[3]
    ]
})
comparison_test['Improvement'] = comparison_test['BERT'] - comparison_test['Baseline (TF-IDF+LR)']
comparison_test['Improvement (%)'] = (comparison_test['Improvement'] * 100).round(2)
print(comparison_test.to_string(index=False))

# 4) Key insights summary
print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)
print(f"\n1. OVERALL PERFORMANCE:")
print(f"   - BERT achieves {bert_test_f1_macro:.1%} test F1 (macro)")
print(f"   - Baseline achieved 71.3% test F1 (macro)")
print(f"   - Improvement: +{(bert_test_f1_macro - 0.7126)*100:.1f} percentage points")

print(f"\n2. CRITICAL CLASS (SUICIDAL):")
suicidal_f1_baseline = 0.7149
suicidal_f1_bert = bert_test_f1_per_class[3]
print(f"   - Baseline Suicidal F1: {suicidal_f1_baseline:.1%}")
print(f"   - BERT Suicidal F1: {suicidal_f1_bert:.1%}")
print(f"   - Improvement: {(suicidal_f1_bert - suicidal_f1_baseline)*100:+.1f} percentage points")

print(f"\n3. DEPRESSION CLASS (HIGH-RISK):")
depression_f1_baseline = 0.5690
depression_f1_bert = bert_test_f1_per_class[2]
print(f"   - Baseline Depression F1: {depression_f1_baseline:.1%}")
print(f"   - BERT Depression F1: {depression_f1_bert:.1%}")
print(f"   - Improvement: {(depression_f1_bert - depression_f1_baseline)*100:+.1f} percentage points")

print(f"\n4. TRIAGE SYSTEM IMPLICATIONS:")
print(f"   - BERT better discriminates high-risk classes (Depression/Suicidal)")
print(f"   - Lower false negatives = fewer missed crisis cases")
print(f"   - Suitable for operational triage priority scoring (Task 3)")

print("\n" + "=" * 80)
print("TASK 1 COMPLETE: Mental-State Classification")
print("=" * 80)
print(f"\n✓ Baseline model trained (TF-IDF + Logistic Regression)")
print(f"✓ BERT model fine-tuned (3 epochs, 89 minutes on T4 GPU)")
print(f"✓ Test evaluation completed")
print(f"✓ Baseline vs BERT comparison generated")
print(f"\nNext: Task 2 - C-SSRS Suicide Risk Severity Scoring")
print("=" * 80)

BERT TEST SET EVALUATION

[1/4] Running BERT inference on test set...


   Test samples: 992
   Predictions shape: (992,)

[2/4] Calculating BERT test metrics...
   Test Accuracy: 0.8871
   Test F1 (macro): 0.8856

   Classification Report (Test):
              precision    recall  f1-score   support

      Normal     0.9228    0.9637    0.9428       248
     Anxiety     0.8898    0.8468    0.8678       248
  Depression     0.8312    0.7742    0.8017       248
    Suicidal     0.8985    0.9637    0.9300       248

    accuracy                         0.8871       992
   macro avg     0.8856    0.8871    0.8856       992
weighted avg     0.8856    0.8871    0.8856       992


   Confusion Matrix (Test):
[[239   2   6   1]
 [  9 210  26   3]
 [ 10  23 192  23]
 [  1   1   7 239]]

[3/4] Creating Baseline vs BERT comparison...

FINAL COMPARISON: BASELINE vs BERT

--- VALIDATION SET PERFORMANCE ---

       Metric  Baseline (TF-IDF+LR)   BERT  Improvement  Improvement (%)
     Accuracy                0.7878 0.8587       0.0709             7.09
     F1 Macro    

In [ ]:
# ===============================
# Save Models & Predictions
# ===============================

import pickle
import os

print("=" * 60)
print("SAVING MODELS & ARTIFACTS")
print("=" * 60)

# Create save directory
SAVE_DIR = "./task1_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) Save BERT model and tokenizer (MOST IMPORTANT - saves 89 minutes!)
print("\n[1/5] Saving BERT model and tokenizer...")
bert_save_path = f"{SAVE_DIR}/bert_mental_health_classifier"
trainer.model.save_pretrained(bert_save_path)
tokenizer.save_pretrained(bert_save_path)
print(f"   ✓ BERT model saved to: {bert_save_path}")
print(f"   ✓ Model size: ~440 MB")

# 2) Save baseline model (TF-IDF + LR)
print("\n[2/5] Saving baseline model (TF-IDF + LR)...")
with open(f"{SAVE_DIR}/baseline_tfidf_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)
with open(f"{SAVE_DIR}/baseline_lr_model.pkl", 'wb') as f:
    pickle.dump(lr_model, f)
print(f"   ✓ Baseline vectorizer saved")
print(f"   ✓ Baseline LR model saved")

# 3) Save test predictions with probabilities (needed for Task 3: Triage)
print("\n[3/5] Saving test predictions and probabilities...")
test_results_df = test_df.copy()
test_results_df['bert_pred_label'] = test_preds
test_results_df['bert_pred_class'] = [id2label[p] for p in test_preds]
for i, label_name in enumerate(LABELS):
    test_results_df[f'bert_prob_{label_name.lower()}'] = test_probs[:, i]

test_results_df.to_csv(f"{SAVE_DIR}/test_predictions_with_probs.csv", index=False)
print(f"   ✓ Test predictions saved with class probabilities")
print(f"   ✓ File: test_predictions_with_probs.csv")

# 4) Save label mappings
print("\n[4/5] Saving label mappings...")
label_mappings = {
    'LABELS': LABELS,
    'label2id': label2id,
    'id2label': id2label
}
with open(f"{SAVE_DIR}/label_mappings.pkl", 'wb') as f:
    pickle.dump(label_mappings, f)
print(f"   ✓ Label mappings saved")

# 5) Save performance summary
print("\n[5/5] Saving performance summary...")
performance_summary = {
    'baseline_val_f1': 0.7715,
    'baseline_test_f1': 0.7091,
    'bert_val_f1': 0.8528,
    'bert_test_f1': bert_test_f1_macro,
    'bert_test_accuracy': bert_test_acc,
    'bert_suicidal_f1': bert_test_f1_per_class[3],
    'bert_depression_f1': bert_test_f1_per_class[2],
    'training_time_minutes': 89.09
}
with open(f"{SAVE_DIR}/performance_summary.pkl", 'wb') as f:
    pickle.dump(performance_summary, f)
print(f"   ✓ Performance metrics saved")

print("\n" + "=" * 60)
print("ALL ARTIFACTS SAVED SUCCESSFULLY!")
print("=" * 60)
print(f"\nSave location: {SAVE_DIR}/")
print("\nSaved files:")
print("  1. bert_mental_health_classifier/ (BERT model + tokenizer)")
print("  2. baseline_tfidf_vectorizer.pkl")
print("  3. baseline_lr_model.pkl")
print("  4. test_predictions_with_probs.csv")
print("  5. label_mappings.pkl")
print("  6. performance_summary.pkl")

print("\n💡 TO RELOAD IN FUTURE SESSION:")
print("\n# Reload BERT model")
print("from transformers import AutoModelForSequenceClassification, AutoTokenizer")
print(f"model = AutoModelForSequenceClassification.from_pretrained('{bert_save_path}')")
print(f"tokenizer = AutoTokenizer.from_pretrained('{bert_save_path}')")
print("\n# Reload baseline")
print("import pickle")
print("with open('task1_models/baseline_tfidf_vectorizer.pkl', 'rb') as f:")
print("    vectorizer = pickle.load(f)")
print("with open('task1_models/baseline_lr_model.pkl', 'rb') as f:")
print("    lr_model = pickle.load(f)")

print("\n=" * 60)

SAVING MODELS & ARTIFACTS

[1/5] Saving BERT model and tokenizer...
   ✓ BERT model saved to: ./task1_models/bert_mental_health_classifier
   ✓ Model size: ~440 MB

[2/5] Saving baseline model (TF-IDF + LR)...
   ✓ Baseline vectorizer saved
   ✓ Baseline LR model saved

[3/5] Saving test predictions and probabilities...
   ✓ Test predictions saved with class probabilities
   ✓ File: test_predictions_with_probs.csv

[4/5] Saving label mappings...
   ✓ Label mappings saved

[5/5] Saving performance summary...
   ✓ Performance metrics saved

ALL ARTIFACTS SAVED SUCCESSFULLY!

Save location: ./task1_models/

Saved files:
  1. bert_mental_health_classifier/ (BERT model + tokenizer)
  2. baseline_tfidf_vectorizer.pkl
  3. baseline_lr_model.pkl
  4. test_predictions_with_probs.csv
  5. label_mappings.pkl
  6. performance_summary.pkl

💡 TO RELOAD IN FUTURE SESSION:

# Reload BERT model
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model = AutoModelForSequenceClassi